In [ ]:
from ipywidgets import widgets
from ipywidgets import Layout

from IPython.display import display
import requests, json, time

from csi_client import CSI_Client


In [ ]:
# debug_view = widgets.Output(layout={'border': '1px solid black'})

# @debug_view.capture(clear_output=True)
# def home_post_action(event):
#     print("debugging technique for silent errors")

# debug_view

In [ ]:
# Location of maestro

maestro_url = 'http://localhost:8090'

SERVICE_URL = maestro_url + '/service/tripplitepdu'

print(f"Service URL is {SERVICE_URL}")

maestro_keys = {'X-API-Key-CSI-Maestro-SystemOperator': 'SystemOperator-1',
                'X-API-Key-CSI-Maestro-Hub': 'Hub-1'}

get_headers = { 'accept':'application/json',
                'Content-Type': 'application/json', 
                **maestro_keys }



In [ ]:
ws_callback_output = widgets.Output(
    value='Websocket Messages',
    placeholder='Async msgs from websocket callback',
    description='Websocket:',
    layout=Layout(width='99%', height='350px', border='2px solid orange'),
    disabled=True
)

# @ws_callback_output.capture(widget)   # Note - capture() does not work on callbacks
def remote_event_handler(payload):
    # This function handles all remote messages coming over the websocket
    # ws_callback_output.flush_output()
    # ws_callback_output.clear_output()
    # time.sleep(3)
    mws_banner = '[ Received message on websocket ]'
    ws_callback_output.append_display_data(mws_banner)
    ws_callback_output.append_display_data(payload)
#    ws_callback_output.append_display('received message')
#    with ws_callback_output:
#        print(mws_banner)
#        print(payload)

In [ ]:
# http response
http_response_output = widgets.Output(
    value='Response from HTTP POST',
    placeholder='Type something',
    description='HTTP\nResponse:',
    layout=Layout(width='99%', height='100px', border='2px solid blue'),
    disabled=True
)

@http_response_output.capture()
def home_post_action(event):
    http_response_output.clear_output()

    _payload = json.dumps(json.loads(json_input_textarea.value))
    print("Issuing HTTP POST with the following payload:")
    print(_payload)

    response = requests.post(SERVICE_URL, json={"payload": _payload}, headers=maestro_keys)
    print(f"response: {response}")
    
def output_clear_both(event):
    # clears both HTTP POST and Websocket output
    http_response_output.clear_output()
    ws_callback_output.clear_output()
    


In [ ]:
def clear_output():
    change_output_button = widgets.Button(description="CLEAR")
    the_output = widgets.Output()
    clear_output_widget = widgets.VBox([change_output_button, the_output])
    clear_output_widget.click_count = 0

    def button_clicked(_button):
        clear_output_widget.click_count += 1
        the_output.clear_output()
        with the_output:
            print(f"button clicked {clear_output_widget.click_count} times.")


    change_output_button.on_click(button_clicked)

    return clear_output_widget


In [ ]:
# Create a CSI_Client, provide a function to handle websocket messages
try:
    rmtconn = CSI_Client(remote_event_handler)
    status = rmtconn.status()
    print(f'remote connection status is {status}')
except:
    print('Could not connect to maestro')


In [ ]:
# Retrieve all DoF data from database
response = requests.get(SERVICE_URL, headers=get_headers)
print('Maestro response:')

print(json.dumps(response.json(),indent=4))

response_json = response.json()


In [ ]:
# Retrieve selected DoF data from database
response = requests.get(SERVICE_URL, headers=get_headers)
print('Filtered Maestro response:\n')

# print(json.dumps(response.json(),indent=4))

response_json = response.json()

# print('\nSelected output from Maestro response:\n')
common = response_json['common']
# print('common:', common)
device = response_json['common']['device']
# print('device:', device)

label = response_json['common']['device']['label']
loc = response_json['common']['device']['location']

print(f"Name (aka Label) is: {label}, Location is: {loc}")

print("\n--- parameters.outlets")
print(response_json['parameters']['outlets'])

print("\n--- sensors.outlets")
print(response_json['sensors']['outlets'])


In [ ]:
# Retrieve selected DoF data from database
response = requests.get(SERVICE_URL, headers=get_headers)
response_json = response.json()
loc = response_json['common']['device']['location']

print(f"Maestro response: {loc}")

# Retrieve location data via SNMP directly
print("\nResponse from SNMP: ",end="")
!MIBS=+TRIPPLITE:TRIPPLITE-PRODUCTS snmpwalk -v 1 -c public 10.0.0.236 TRIPPLITE-PRODUCTS::tlpDeviceLocation


In [ ]:
# json input area
json_input_textarea = widgets.Textarea(
    value='',
    placeholder='{parameters: ...}',
    description='String:',
    layout=Layout(width='75%', height='100px'),
    disabled=False
)
json_input_textarea.description = "JSON"

# post button
home_post_button = widgets.Button(
    description='HTTP POST',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Update DoF values in JSON',
    layout=Layout(position='right'),
    icon='' # (FontAwesome names without the `fa-` prefix)
)
home_post_button.on_click(home_post_action)

# clear button
output_clear_button = widgets.Button(
    description='CLEAR OUTPUT',
    disabled=False,
    button_style='warning', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Clear output',
    layout=Layout(position='right'),
    icon='' # (FontAwesome names without the `fa-` prefix)
)
output_clear_button.on_click(output_clear_both)

top_right_box = widgets.VBox([home_post_button, output_clear_button])
top_box = widgets.HBox([json_input_textarea, top_right_box])

http_response_output_label = widgets.Label(value='HTTP Post Results')
ws_callback_output_label = widgets.Label(value='Websocket Callback Msgs')

bottom_box = widgets.VBox([http_response_output_label, http_response_output, 
                           ws_callback_output_label, ws_callback_output])
# bottom_box = widgets.VBox([http_response_output, ws_callback_output])

app_box = widgets.VBox([top_box, bottom_box])

app_box

# Some sample DoF state changes in json 
# Note: Use Ctrl-C and Ctrl-V to cut and paste in Jupyter

# {"proxy_configuration": {"comms": {"ip": "10.0.0.236", 
#  "read_community": "public", "write_community": "private", "snmp_version": 1}}}

# {"common": {"device": {"label": "SamplePDUName"}}}
# {"common": {"device": {"location": "Westford"}}}
# {"parameters": {"outlets": {"1.4": {"state": "2"}}}}


In [ ]:
# Retrieve selected DoF data from database
response = requests.get(SERVICE_URL, headers=get_headers)
response_json = response.json()
loc = response_json['common']['device']['location']

print(f"Maestro response: {loc}")

# Retrieve location data via SNMP directly
print("\nResponse from SNMP: ",end="")
!MIBS=+TRIPPLITE:TRIPPLITE-PRODUCTS snmpwalk -v 1 -c public 10.0.0.236 TRIPPLITE-PRODUCTS::tlpDeviceLocation


In [ ]:
# Walk entire MIB
# Have to build net-snmp tools on C2 machine
# TRIPPLITE mibs must be copied to /usr/local/share/snmp/mibs
!MIBS=+TRIPPLITE:TRIPPLITE-PRODUCTS snmpwalk -v 1 -c public 10.0.0.236 1


In [ ]:
# Misc tests
#!MIBS=+TRIPPLITE:TRIPPLITE-PRODUCTS snmpwalk -v 1 -c public 10.0.0.236 TRIPPLITE-PRODUCTS::tlpDeviceName
#!MIBS=+TRIPPLITE:TRIPPLITE-PRODUCTS snmpwalk -v 1 -c public 10.0.0.236 sysName
